# POLG rare variant extraction from WGS data

- **Project**: Multi-ancestry analysis of POLG variants in Parkinson’s disease
- **Last Update:** APRIL-2026

## Set path

In [ ]:
import pathlib
REL11_PATH = pathlib.Path(pathlib.Path.home(), "/path/to/gp2/release11")
!ls -hal {REL11_PATH}

In [ ]:
## Define file paths for clinical and genetic data
EXTENDED_CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/extended_clinical_data_release11_vwb.csv')
CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/master_key_release11_final_vwb.csv')

## Install packages

In [ ]:
%%capture
%%bash

#To install plink 1.9
cd /home/jupyter/
if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter/"
else
    echo "Plink is not installed"
    cd /home/jupyter

    wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

In [ ]:
%%bash

#chmod plink 1.9 to ensure permission to run the program
chmod u+x /home/jupyter/plink1.9

In [ ]:
%%capture
%%bash

#To install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

unzip -o plink2_linux_x86_64_latest.zip

fi

In [ ]:
%%bash

#chmod plink 2 to ensure permission to run the program
chmod u+x /home/jupyter/plink2

In [ ]:
%%bash 

#install bcftools

#mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/bcftools; then
    echo "bcftools is already installed in /home/jupyter/tools/"
else
    echo -e "Downloading bcftools \n    -------"
    git clone --recurse-submodules https://github.com/samtools/htslib.git
    git clone https://github.com/samtools/bcftools.git
    cd bcftools
    make
    echo -e "\n bcftools downloaded and unzipped in /home/jupyter/tools \n "

fi





In [ ]:
# check bcftools 
!/home/jupyter/tools/bcftools/bcftools --help

## POLG variant extraction

In [ ]:
%%bash
# Create a folder on your workspace
mkdir /home/jupyter/POLG_results/vcfswgs
## Define the working directory
cd /home/jupyter/POLG_results/vcfswgs

In [ ]:
# Create vcf files from binary files instead
# Create a sub-folder on your workspace for the ancestry  files
mkdir /home/jupyter/POLG_results/vcfswgs
ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']
for ancestry in ancestries:
    
    ! /home/jupyter/plink2 \
    --pfile "/path/to/gp2/release11/wgs/deepvariant_joint_calling/plink/{ancestry}/chr15_{ancestry}_release11" \
    --chr 15 \
    --from-bp 89305198 \
    --to-bp 89334861 \
    --make-bed \
    --out /home/jupyter/POLG_results/vcfswgs/{ancestry}_POLG

In [ ]:
for ancestry in ancestries:
        
    #To turn binary files into VCF
    ! /home/jupyter/plink2 \
    --bfile /home/jupyter/POLG_results/vcfswgs/{ancestry}_POLG \
    --recode vcf \
    --out /home/jupyter/POLG_results/vcfswgs/{ancestry}_POLG

In [ ]:
ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']
for ancestry in ancestries:  
    ! bgzip -f /home/jupyter/POLG_results/vcfswgs/{ancestry}_POLG.vcf

    ! tabix -f /home/jupyter/POLG_results/vcfswgs/{ancestry}_POLG.vcf.gz

## Annotation

In [ ]:
# annotate with annovar
# Create a sub-folder for the annotated files
mkdir /home/jupyter/POLG_results/vcfswgs/vcfs
ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']
for ancestry in ancestries:
    ! perl /home/jupyter/annovar/table_annovar.pl /home/jupyter/POLG_results/vcfswgs/{ancestry}_POLG.vcf.gz /home/jupyter/tools/annovar/humandb/ -buildver hg38 \
    -out /home/jupyter/POLG_results/vcfswgs/vcfs/{ancestry}_POLG.annovar \
    -remove -protocol refGene,gnomad41_genome,clinvar_20250721,dbnsfp47a \
    -operation g,f,f,f \
    --nopolish \
    -nastring . \
    -vcfinput

In [ ]:
# Select the columns to keep
basic_cols= anno.columns.tolist()[0:10]
additional_cols_to_keep=['Otherinfo6',
                         'gnomad41_genome_AF',
                         'gnomad41_genome_fafmax_faf95_max',
                         'CLNDN',
                         'CLNSIG',
                         'CADD_phred']
            
all_cols_to_keep= basic_cols+ additional_cols_to_keep
all_cols_to_keep

In [ ]:
df=anno[all_cols_to_keep]

# Rename column
df.rename({'Otherinfo6':'var_id'},axis=1,inplace=True)

df

In [ ]:
# Count occurrences of each value
value_counts = df["ExonicFunc.refGene"].value_counts()

# Display counts
print(value_counts)

In [ ]:
# Define the values you want to keep
keep_values = ["exonic", "splicing", "exonic;splicing"]  # Add more if needed

# Subset the dataframe
filtered_df = df[df["Func.refGene"].isin(keep_values)]

# Display the filtered dataframe
print (filtered_df.head())

In [ ]:
# Filter out synonymous SNVs
filtered_df = filtered_df[filtered_df["ExonicFunc.refGene"] != "synonymous SNV"]

# Display the filtered dataframe
print (filtered_df.head())

In [ ]:
# Save the filtered output
filtered_df.to_csv('/home/jupyter/POLG_results/vcfswgs/vcfs/{ancestry}_filtered_multianno.tsv', sep="\t", index=False)

# Write out 'var_id' to extract from plink files
filtered_df['var_id'].to_csv('/home/jupyter/POLG_results/vcfswgs/vcfs/{ancestry}_var_to_extract.txt',index=False,header=False)

In [ ]:
! /home/jupyter/plink2 \
    --bfile /home/jupyter/POLG_results/vcfswgs/{ancestry}_POLG \
    --extract  /home/jupyter/POLG_results/vcfswgs/vcfs/{ancestry}_var_to_extract.txt \
    --recode A \
    --out /home/jupyter/POLG_results/vcfswgs/vcfs/{ancestry}_POLG


## Examination of carriers

In [ ]:
wgs_var = pd.read_csv('/home/jupyter/POLG_results/vcfswgs/vcfs/{ancestry}_POLG.raw', sep='\s+')
wgs_var

In [ ]:
# Transpose the dataframe to be row as variants and columns as samples
var_col=wgs_var.columns[6:len(wgs_var)]
d = wgs_var.drop(columns=['FID','PAT','MAT','SEX'])
sample=wgs_var[['IID','PHENOTYPE']]

# Filtering rows where any value in 'var_col' is ≤1 (either het or hom)
t=d[(d[var_col]<=1).any(axis=1)].T
t.columns = t.iloc[0]
t=t.iloc[1:]
t.reset_index(inplace=True)

t

In [ ]:
# Strip the last '_${ref_allele}', so we can keep the same variant id as in annotation 
t['index'] = t['index'].str.rsplit('_', n=1).str[0]
t.rename({'index':'var_id'},axis=1,inplace=True)
t

In [ ]:
t['hom_carrier'] = t.apply(lambda row: row[row == 0].index.tolist() , axis=1)
t['het_carrier'] = t.apply(lambda row: row[row == 1].index.tolist() , axis=1)  

# Store hom and het seperately to later explode the dataframe correctly
hom = t[['var_id','hom_carrier']]

In [ ]:
hom = hom.explode('hom_carrier', ignore_index=True)
hom= hom.loc[~hom['hom_carrier'].isnull()]

hom

In [ ]:
# Merge with annotation
out_hom = pd.merge(hom,filtered_df, on='var_id',how='left')
out_hom['zygosity'] = 'hom'

# Rename column
out_hom.rename({'hom_carrier':'carrier_id'},axis=1,inplace=True)

out_hom

In [ ]:
# Repeat the same for het
het = t[['var_id','het_carrier']]
het = het.explode('het_carrier', ignore_index=True)
het= het.loc[~het['het_carrier'].isnull()]

het

In [ ]:
# Merge with annotation
out_het = pd.merge(het,filtered_df, on='var_id',how='left')
out_het['zygosity'] = 'het'

# Rename col
out_het.rename({'het_carrier':'carrier_id'},axis=1,inplace=True)

out_het

In [ ]:
# Check if there's any comphet by grouping gene and sample ID
pd.concat(g for _, g in out_het.groupby(["Gene.refGene","carrier_id"]) if len(g) > 1)

In [ ]:
# Get everything and write out
merged_df_wgs = pd.concat([out_het,out_hom],axis=0)

# Save the dataset
merged_df_wgs.to_csv("/home/jupyter/POLG_results/vcfswgs/vcfs/{ancestry}_merged_genotypes_wgs_POLG.tsv",sep='\t',index=False)

In [ ]:
import pandas as pd
import os

ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']
all_dfs = []

for ancestry in ancestries:
    file_path = f'/home/jupyter/POLG_results/vcfswgs/vcfs/{ancestry}_merged_genotypes_wgs_POLG.tsv'
    
    # Check if the file exists before attempting to read
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, sep='\t')
        
        # Double check that the Ancestry column is present; if not, add it
        if 'Ancestry' not in df.columns:
            df['Ancestry'] = ancestry
            
        all_dfs.append(df)
        print(f"Loaded {ancestry}: {len(df)} rows")
    else:
        print(f"Warning: File for {ancestry} not found at {file_path}")

# Perform the vertical merge (binding rows)
if all_dfs:
    final_merged_df = pd.concat(all_dfs, axis=0, ignore_index=True)
    
    # Save the consolidated file
    output_path = "/home/jupyter/POLG_results/vcfswgs/vcfs/FINAL_CONSOLIDATED_genotypes_POLG.tsv"
    final_merged_df.to_csv(output_path, sep='\t', index=False)
    
    print("\n--- Merge Complete ---")
    print(f"Total rows in final file: {len(final_merged_df)}")
    print(f"Final file saved to: {output_path}")
else:
    print("No files were found to merge.")